Cell 0: setup

In [45]:
!pip install -q requests pandas geopandas

import os, re, time
import requests
import pandas as pd
import geopandas as gpd

DATA_DIR = "data_raw"
os.makedirs(DATA_DIR, exist_ok=True)

CENSUS_KEY = None  # free key: api.census.gov/data/key_signup.html  ->  CENSUS_KEY = "your_key"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "boston-gas-pipe-triage (portfolio project)"})

def get_json(url, params=None, retries=3):
    for i in range(retries):
        r = SESSION.get(url, params=params, timeout=90)
        if r.status_code == 200:
            try:
                return r.json()
            except ValueError:
                raise ValueError(f"Not JSON from {r.url}\n--- server said ---\n{r.text[:400]}")
        time.sleep(2 * (i + 1))
    r.raise_for_status()

def path(name):
    return os.path.join(DATA_DIR, name)

def already_have(name):
    if os.path.exists(path(name)):
        print(f"Using cached {name}")
        return True
    return False

Cell 1: search Boston's open data portal (run this first and look at the output)

In [46]:
CKAN = "https://data.boston.gov/api/3/action"
CKAN_COLS = ["dataset", "resource", "format", "resource_id", "url", "datastore", "modified"]

def ckan_search(query, rows=10):
    results = get_json(f"{CKAN}/package_search", {"q": query, "rows": rows})["result"]["results"]
    out = []
    for pkg in results:
        for rsc in pkg.get("resources", []):
            out.append({
                "dataset": pkg["title"],
                "resource": rsc.get("name"),
                "format": (rsc.get("format") or "").upper(),
                "resource_id": rsc["id"],
                "url": rsc.get("url"),
                "datastore": rsc.get("datastore_active", False),
                "modified": rsc.get("last_modified"),
            })
    return pd.DataFrame(out, columns=CKAN_COLS)

for q in ["street segments", "property assessment"]:
    print(f"\n=== {q} ===")
    display(ckan_search(q)[["dataset", "resource", "format", "datastore", "modified"]].head(8))


=== street segments ===


,dataset,resource,format,datastore,modified
0,Boston Street Segments Metadata,ArcGIS Hub Dataset,HTML,False,NaN
1,Boston Street Segments (SAM System),GeoJSON,GEOJSON,False,2026-10-05T03:01:26.840400
2,Boston Street Segments (SAM System),CSV,CSV,True,2026-10-05T03:01:20.310639
3,Boston Street Segments (SAM System),ArcGIS GeoServices REST API,ARCGIS GEOSERVICES REST API,False,2026-10-04T23:01:00
4,Boston Street Segments (SAM System),ArcGIS Hub Dataset,HTML,False,NaN
5,Boston Street Segments (SAM System),KML,KML,False,2025-10-09T03:02:42.280072
6,Boston Street Segments (SAM System),Shapefile,SHP,False,2025-09-30T03:03:06.183312



=== property assessment ===


,dataset,resource,format,datastore,modified
0,Property Assessment,NaN,PDF,False,2021-04-23T18:52:08.855785
1,Property Assessment,NaN,CSV,True,2026-05-22T15:39:06.823803
2,Property Assessment,NaN,PDF,False,NaN
3,Property Assessment,NaN,CSV,True,NaN
4,Property Assessment,NaN,PDF,False,NaN
5,Property Assessment,NaN,CSV,True,2024-01-09T20:10:02.540265
6,Property Assessment,NaN,PDF,False,2024-01-09T20:08:16.912733
7,Property Assessment,NaN,CSV,True,2023-01-31T15:30:38.833714


Cell 2: street segments and latest property assessment

In [47]:
# Property assessment: most recently updated CSV from the "Property Assessment" dataset only
if os.path.exists(path("boston_property_assessment.csv")):
    if pd.read_csv(path("boston_property_assessment.csv"), nrows=5).shape[1] < 30:
        os.remove(path("boston_property_assessment.csv"))
        print("Removed the wrong file downloaded earlier")

if not already_have("boston_property_assessment.csv"):
    pa = ckan_search("property assessment", rows=10)
    pa = pa[pa["dataset"].eq("Property Assessment") & pa["format"].eq("CSV")]
    pa = pa.dropna(subset=["modified"]).sort_values("modified", ascending=False)
    latest = pa.iloc[0]
    print("Downloading CSV last modified:", latest["modified"], "\n", latest["url"])
    download(latest["url"], "boston_property_assessment.csv")
assess = pd.read_csv(path("boston_property_assessment.csv"), low_memory=False)
print("Assessment rows:", assess.shape)
print("Useful columns:", [c for c in assess.columns if re.search(r"YR_|HEAT|FUEL|LU|BLDG|UNIT|ST_|ZIP", c, re.I)])

Using cached boston_property_assessment.csv
Assessment rows: (184552, 67)
Useful columns: ['ST_NUM', 'ST_ALPHA', 'ST_NUM2', 'ST_NAME', 'UNIT_NUM', 'ZIP_CODE', 'BLDG_SEQ', 'NUM_BLDGS', 'LUC', 'LU', 'LU_DESC', 'BLDG_TYPE', 'MAIL_ZIP_CODE', 'RES_UNITS', 'COM_UNITS', 'RC_UNITS', 'LAND_VALUE', 'BLDG_VALUE', 'SFYI_VALUE', 'TOTAL_VALUE', 'YR_BUILT', 'YR_REMODEL', 'HEAT_TYPE', 'HEAT_SYSTEM', 'CORNER_UNIT']


Cell 3: Census ACS by block group (Suffolk County)

In [48]:
# ACS 5-year tables from the Census bulk "Summary File" (no API key needed)
ACS_GROUPS = {
    "B25040": "heating fuel",
    "B25034": "year built",
    "B19013": "median income",
    "B25003": "tenure",
}
SF_URL = "https://www2.census.gov/programs-surveys/acs/summary_file/{y}/table-based-SF/data/5YRData/acsdt5y{y}-{t}.dat"
KEEP_PREFIX = "1500000US25025"   # block groups in Suffolk County, MA

def acs_from_summary_file(group, years=(2024, 2023)):
    for year in years:
        url = SF_URL.format(y=year, t=group.lower())
        try:
            with SESSION.get(url, stream=True, timeout=600) as r:
                if r.status_code != 200:
                    print(f"ACS {year} {group}: not available (status {r.status_code})")
                    continue
                lines = (ln.decode("utf-8", errors="replace") if isinstance(ln, bytes) else ln
                         for ln in r.iter_lines())
                header = next(lines).split("|")
                rows = [ln.split("|") for ln in lines if ln.startswith(KEEP_PREFIX)]
        except requests.RequestException as e:
            print(f"ACS {year} {group}: download failed: {e}")
            continue
        df = pd.DataFrame(rows, columns=header)
        df["GEOID"] = df["GEO_ID"].str.replace("1500000US", "", regex=False)
        est = [c for c in df.columns if re.fullmatch(rf"{group}_E\d{{3}}", c)]
        df = df[["GEOID"] + est]
        df.columns = ["GEOID"] + [f"{group}_{c[-3:]}E" for c in est]   # B25040_E002 -> B25040_002E
        num = df.columns[1:]
        df[num] = df[num].apply(pd.to_numeric, errors="coerce").where(lambda x: x >= 0)
        print(f"{group} ({ACS_GROUPS[group]}): ACS {year} 5-year, {len(df)} block groups")
        return df
    raise RuntimeError(f"Could not fetch {group}; see the messages above")

if not already_have("acs_suffolk_blockgroups.csv"):
    acs = None
    for g in ACS_GROUPS:
        df = acs_from_summary_file(g)
        acs = df if acs is None else acs.merge(df, on="GEOID")

    acs["pct_gas_heat"]  = acs["B25040_002E"] / acs["B25040_001E"]
    acs["pct_elec_heat"] = acs["B25040_004E"] / acs["B25040_001E"]
    acs["pct_pre1940"]   = acs["B25034_011E"] / acs["B25034_001E"]
    acs["pct_renter"]    = acs["B25003_003E"] / acs["B25003_001E"]
    acs = acs.rename(columns={"B19013_001E": "median_hh_income"})
    acs.to_csv(path("acs_suffolk_blockgroups.csv"), index=False)

acs = pd.read_csv(path("acs_suffolk_blockgroups.csv"), dtype={"GEOID": str})
acs[["pct_gas_heat", "pct_pre1940", "pct_renter", "median_hh_income"]].describe()

Using cached acs_suffolk_blockgroups.csv


,pct_gas_heat,pct_pre1940,pct_renter,median_hh_income
count,660.000000,661.000000,660.000000,552.000000
mean,0.591146,0.477934,0.620081,112560.351449
std,0.199112,0.249687,0.257448,55951.318233
min,0.000000,0.000000,0.000000,2499.000000
25%,0.461967,0.291066,0.445444,72622.750000
50%,0.612965,0.482993,0.633785,106450.500000
75%,0.733450,0.685015,0.837943,144617.000000
max,1.000000,1.000000,1.000000,250001.000000


Cell 4: Census block group boundaries

In [49]:
def arcgis_query_all(url, where="1=1", bbox=None, out_fields="*", page=2000):
    feats, offset = [], 0
    while True:
        params = {"where": where, "outFields": out_fields, "f": "geojson", "outSR": 4326,
                  "resultOffset": offset, "resultRecordCount": page}
        if bbox:
            params.update({"geometry": bbox, "geometryType": "esriGeometryEnvelope",
                           "inSR": 4326, "spatialRel": "esriSpatialRelIntersects"})
        batch = get_json(url, params).get("features", [])
        feats += batch
        if len(batch) < page:
            break
        offset += page
    return gpd.GeoDataFrame.from_features(feats, crs="EPSG:4326")

BG_URL = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/Tracts_Blocks/MapServer/1/query"

if not already_have("suffolk_blockgroups.geojson"):
    bg = arcgis_query_all(BG_URL, where="STATE='25' AND COUNTY='025'", out_fields="GEOID")
    bg.to_file(path("suffolk_blockgroups.geojson"), driver="GeoJSON")
bg = gpd.read_file(path("suffolk_blockgroups.geojson"))
print("Block groups:", bg.shape)
print("GEOIDs matching ACS:", bg["GEOID"].isin(acs["GEOID"]).sum())

Using cached suffolk_blockgroups.geojson
Block groups: (681, 2)
GEOIDs matching ACS: 681


Cell 5: MassGIS 2020 environmental-justice block groups

In [50]:
EJ_URL = ("https://services.arcgis.com/c5WwApDsDjRhIVkH/ArcGIS/rest/services/"
          "Environmental_Justice_2020_Populations/FeatureServer/0/query")
BOSTON_BBOX = "-71.1912,42.2279,-70.9860,42.3974"

if not already_have("ej_2020_boston_area.geojson"):
    ej = arcgis_query_all(EJ_URL, bbox=BOSTON_BBOX)
    ej.to_file(path("ej_2020_boston_area.geojson"), driver="GeoJSON")
ej = gpd.read_file(path("ej_2020_boston_area.geojson"))
print("EJ block groups:", ej.shape)
print(ej.columns.tolist())

Using cached ej_2020_boston_area.geojson
EJ block groups: (770, 27)
['total_hh', 'ej', 'objectid', 'geo_name', 'geoid', 'splitmuni', 'muni', 'total_pop', 'nh_white', 'pct_mnrty', 'lim_eng', 'limengpct', 'mhhi', 'mhhi_pct', 'mhhi_2019', 'muni_mhhi', 'munmamhhi', 'ej_mhhi', 'ej_limeng', 'ej_mnrty', 'ej_crit', 'ej_crcnt', 'ej_crdesc', 'Shape__Area', 'Shape__Length', 'GlobalID', 'geometry']


Cell 6: daily weather since 2000 (Open-Meteo, no key needed)

In [51]:
if not already_have("boston_daily_weather.csv"):
    end = (pd.Timestamp.today() - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
    wx = get_json("https://archive-api.open-meteo.com/v1/archive", {
        "latitude": 42.3601, "longitude": -71.0589,
        "start_date": "2000-01-01", "end_date": end,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "America/New_York",
    })
    w = pd.DataFrame(wx["daily"])
    w.to_csv(path("boston_daily_weather.csv"), index=False)

weather = pd.read_csv(path("boston_daily_weather.csv"), parse_dates=["time"])
weather["freeze_thaw"] = (weather["temperature_2m_min"] < 0) & (weather["temperature_2m_max"] > 0)
weather["hdd_65F"] = (18.33 - (weather["temperature_2m_max"] + weather["temperature_2m_min"]) / 2).clip(lower=0)

annual_wx = weather.groupby(weather["time"].dt.year).agg(
    freeze_thaw_days=("freeze_thaw", "sum"), hdd=("hdd_65F", "sum"), precip_mm=("precipitation_sum", "sum"))
annual_wx.tail(10)

Using cached boston_daily_weather.csv


,freeze_thaw_days,hdd,precip_mm
time,,,
2017,85,3157.53,1175.7
2018,93,3300.26,1477.3
2019,91,3360.69,1325.9
2020,87,3011.95,1047.6
2021,92,3018.04,1390.8
2022,76,3100.12,1027.5
2023,87,2898.85,1425.5
2024,77,2965.56,1204.3
2025,69,3274.92,1021.8


Cell 7: check what you have

In [52]:
if os.path.exists(path("boston_street_segments.geojson")):
    streets = gpd.read_file(path("boston_street_segments.geojson"))
    print("Street segments:", streets.shape)
    print("Street segment columns:", streets.columns.tolist())
else:
    print("Street segments file is missing. Rerun Cell 2.")
print()
for f in sorted(os.listdir(DATA_DIR)):
    print(f"{f:45s} {os.path.getsize(path(f)) / 1e6:8.1f} MB")

Street segments: (19440, 40)
Street segment columns: ['OBJECTID', 'SEGMENT_ID', 'L_F_ADD', 'L_T_ADD', 'R_F_ADD', 'R_T_ADD', 'STREET_ID', 'PRE_DIR', 'ST_NAME', 'ST_TYPE', 'SUF_DIR', 'ALTERNATE_NAME', 'CFCC', 'SPEEDLIMIT', 'ONEWAY', 'F_ZLEV', 'T_ZLEV', 'FT_COST', 'TF_COST', 'FT_DIR', 'TF_DIR', 'SHIELD', 'HWY_NUM', 'MUN_L', 'MUN_R', 'NBHD_L', 'NBHD_R', 'STATE00_L', 'STATE00_R', 'COUNTY00_L', 'COUNTY00_R', 'MCD00_L', 'MCD00_R', 'ZIP_L', 'ZIP_R', 'ownership', 'ROUTE_ID', 'Shape_Length', 'shape_wkt', 'geometry']

acs_suffolk_blockgroups.csv                        0.1 MB
berdo_2025_check.xlsx                              1.6 MB
berdo_latest.csv                                   1.6 MB
berdo_source.txt                                   0.0 MB
boston_daily_weather.csv                           0.2 MB
boston_property_assessment.csv                    80.2 MB
boston_street_segments.geojson                    22.4 MB
ej_2020_boston_area.geojson                        2.6 MB
suffolk_blockgroups.geo

Cell 8: clean the street segments

In [53]:
import numpy as np

PROC_DIR = "data_processed"
os.makedirs(PROC_DIR, exist_ok=True)
MA_CRS = "EPSG:26986"   # Massachusetts State Plane, meters

SUFFIX = {"AVENUE": "AV", "AVE": "AV", "AV": "AV", "STREET": "ST", "ST": "ST", "ROAD": "RD", "RD": "RD",
          "BOULEVARD": "BL", "BLVD": "BL", "BL": "BL", "PLACE": "PL", "PL": "PL", "TERRACE": "TER",
          "TER": "TER", "TE": "TER", "COURT": "CT", "CT": "CT", "DRIVE": "DR", "DR": "DR", "LANE": "LA",
          "LN": "LA", "LA": "LA", "SQUARE": "SQ", "SQ": "SQ", "PARKWAY": "PKWY", "PKWY": "PKWY", "PW": "PKWY",
          "HIGHWAY": "HWY", "HWY": "HWY", "WAY": "WAY", "WY": "WAY", "CIRCLE": "CIR", "CIR": "CIR",
          "CI": "CIR", "PARK": "PARK", "PK": "PARK", "ROW": "ROW", "WHARF": "WHF", "WHF": "WHF",
          "ALLEY": "AL", "AL": "AL", "HILL": "HL", "HL": "HL"}
DIRS = {"NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W", "N": "N", "S": "S", "E": "E", "W": "W"}

def street_key(*parts):
    """Standardize a street name so 'Commonwealth Avenue' and 'COMMONWEALTH AV' match."""
    text = " ".join(str(p) for p in parts if pd.notna(p) and str(p).strip())
    tokens = re.sub(r"[^A-Z0-9 ]", " ", text.upper()).split()
    if not tokens:
        return None
    tokens = [DIRS.get(t, t) if i in (0, len(tokens) - 1) else t for i, t in enumerate(tokens)]
    for i in range(len(tokens) - 1, 0, -1):
        if tokens[i] in SUFFIX:
            tokens[i] = SUFFIX[tokens[i]]
            break
        if tokens[i] not in DIRS.values():
            break
    return " ".join(tokens)

seg = gpd.read_file(path("boston_street_segments.geojson")).to_crs(MA_CRS)
seg["seg_id"] = seg["SEGMENT_ID"] if seg["SEGMENT_ID"].is_unique else seg["OBJECTID"]
seg["st_key"] = [street_key(a, b, c, d) for a, b, c, d in
                 zip(seg["PRE_DIR"], seg["ST_NAME"], seg["ST_TYPE"], seg["SUF_DIR"])]
seg["length_m"] = seg.geometry.length

# House-number range on each segment (0 means no address)
for c in ["L_F_ADD", "L_T_ADD", "R_F_ADD", "R_T_ADD"]:
    seg[c] = pd.to_numeric(seg[c], errors="coerce")
rng = seg[["L_F_ADD", "L_T_ADD", "R_F_ADD", "R_T_ADD"]].replace(0, np.nan)
seg["addr_min"], seg["addr_max"] = rng.min(axis=1), rng.max(axis=1)

# Keep surface streets where gas mains run: CFCC A2-A4 roads and A73 alleys
keep = seg["CFCC"].astype(str).str.match(r"^A[2-4]|^A73")
if keep.mean() < 0.3:
    print("Warning: CFCC filter would drop most segments; keeping all")
    keep[:] = True
seg = seg[keep & seg["st_key"].notna() & (seg["length_m"] > 1)].copy()

seg = seg[["seg_id", "st_key", "ST_NAME", "ST_TYPE", "CFCC", "NBHD_L",
           "addr_min", "addr_max", "length_m", "geometry"]]
print(f"Street segments kept: {len(seg):,}  |  total length: {seg['length_m'].sum() / 1609.34:,.0f} miles")
print("Segments with house numbers:", seg["addr_min"].notna().mean().round(3))

Street segments kept: 17,820  |  total length: 1,032 miles
Segments with house numbers: 0.902


Cell 9: attach buildings to each segment (property assessment)

In [54]:
a = pd.read_csv(path("boston_property_assessment.csv"), low_memory=False)
a = a[a["LU"] != "CM"].copy()   # drop condo "master" rows so units aren't double counted

print("Land-use codes:\n", a["LU"].value_counts().head(12), "\n")
print("Heat types:\n", a["HEAT_TYPE"].value_counts(dropna=False).head(12), "\n")
print("Heat systems:\n", a["HEAT_SYSTEM"].value_counts(dropna=False).head(12), "\n")

a["st_num"] = pd.to_numeric(a["ST_NUM"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
a["st_key"] = a["ST_NAME"].map(lambda s: street_key(s))
a["yr_built"] = pd.to_numeric(a["YR_BUILT"], errors="coerce").where(lambda y: y.between(1700, 2026))

# Heat pumps: search both heat columns for common spellings
heat_text = a["HEAT_TYPE"].astype(str) + " " + a["HEAT_SYSTEM"].astype(str)
a["heat_pump"] = heat_text.str.contains(r"pump|ht\s*pmp|\bhp\b|mini.?split", case=False, regex=True)

# Housing units: use the reported count, otherwise a typical count for the land-use code
LU_UNITS = {"R1": 1, "R2": 2, "R3": 3, "R4": 5, "A": 8, "CD": 1, "RC": 2}
reported = pd.to_numeric(a["RES_UNITS"].astype(str).str.replace(",", ""), errors="coerce")
a["res_units"] = reported.where(reported > 0, a["LU"].map(LU_UNITS)).fillna(0)

a["total_value"] = pd.to_numeric(a["TOTAL_VALUE"].astype(str).str.replace(r"[,$]", "", regex=True), errors="coerce")

# One row per street address
addr = (a.dropna(subset=["st_num", "st_key"])
          .groupby(["st_key", "st_num"])
          .agg(n_parcels=("LU", "size"), res_units=("res_units", "sum"), yr_built=("yr_built", "median"),
               heat_pump=("heat_pump", "max"), total_value=("total_value", "sum"))
          .reset_index())

# Match each address to the segment whose house-number range contains it
segr = seg.dropna(subset=["addr_min"])[["seg_id", "st_key", "addr_min", "addr_max"]].sort_values("addr_min")
m = pd.merge_asof(addr.sort_values("st_num"), segr, left_on="st_num", right_on="addr_min",
                  by="st_key", direction="backward")
m = m[m["st_num"] <= m["addr_max"]]
print(f"Addresses matched to a street segment: {len(m) / len(addr):.1%} of {len(addr):,}")

bldg = (m.groupby("seg_id")
          .agg(n_addresses=("st_num", "size"), n_parcels=("n_parcels", "sum"), res_units=("res_units", "sum"),
               med_yr_built=("yr_built", "median"), pct_heat_pump=("heat_pump", "mean"),
               total_value=("total_value", "sum"))
          .reset_index())
seg = seg.drop(columns=[c for c in bldg.columns if c != "seg_id" and c in seg.columns]).merge(bldg, on="seg_id", how="left")

# Segments with no matched buildings: zero counts; borrow building age from the same street, then neighborhood
for c in ["n_addresses", "n_parcels", "res_units", "total_value"]:
    seg[c] = seg[c].fillna(0)
seg["yr_built_imputed"] = seg["med_yr_built"].isna()
seg["med_yr_built"] = (seg["med_yr_built"]
                       .fillna(seg.groupby("st_key")["med_yr_built"].transform("median"))
                       .fillna(seg.groupby("NBHD_L")["med_yr_built"].transform("median"))
                       .fillna(seg["med_yr_built"].median()))
seg["pct_heat_pump"] = seg["pct_heat_pump"].fillna(0)
print(f"Parcels flagged as heat pump: {a['heat_pump'].mean():.2%}")
seg[["n_addresses", "res_units", "med_yr_built", "pct_heat_pump"]].describe().round(2)

Land-use codes:
 LU
CD         74254
R1         30441
R2         16701
R3         13343
CP          8545
E           7572
RL - RL     5854
C           4566
A           3116
RC          2956
R4          2485
CC          1551
Name: count, dtype: int64 

Heat types:
 HEAT_TYPE
W - Ht Water/Steam    69740
F - Forced Hot Air    54522
NaN                   37762
E - Electric           5978
P - Heat Pump          4734
S - Space Heat          561
N - None                 83
O - Other                33
Name: count, dtype: int64 

Heat systems:
 HEAT_SYSTEM
NaN                   98904
I - Indiv. Cntrl      46230
Y - Self Contained    19243
C - Common             8012
N - None                966
1                        58
Name: count, dtype: int64 

Addresses matched to a street segment: 96.0% of 90,790
Parcels flagged as heat pump: 2.58%


,n_addresses,res_units,med_yr_built,pct_heat_pump
count,17820.00,17820.00,17820.00,17820.00
mean,4.89,11.73,1923.94,0.00
std,7.07,23.75,30.43,0.05
min,0.00,0.00,1752.00,0.00
25%,0.00,0.00,1903.00,0.00
50%,2.00,3.00,1915.00,0.00
75%,7.00,14.00,1935.00,0.00
max,68.00,525.00,2025.00,1.00


Cell 10: attach Census and environmental-justice data

In [55]:
bg = gpd.read_file(path("suffolk_blockgroups.geojson")).to_crs(MA_CRS)
acs = pd.read_csv(path("acs_suffolk_blockgroups.csv"), dtype={"GEOID": str})
print("Block groups with ACS data:", bg["GEOID"].isin(acs["GEOID"]).sum(), "of", len(bg))

# Fill suppressed Census values with the tract median, then the county median; flag income caps
acs_cols = ["median_hh_income", "pct_gas_heat", "pct_elec_heat", "pct_pre1940", "pct_renter"]
acs["income_imputed"] = acs["median_hh_income"].isna()
acs["income_topcoded"] = acs["median_hh_income"] >= 250000
acs["tract"] = acs["GEOID"].str[:11]
for c in acs_cols:
    acs[c] = acs[c].fillna(acs.groupby("tract")[c].transform("median")).fillna(acs[c].median())
acs = acs.rename(columns={"B25040_001E": "bg_households"})

# Each segment's midpoint -> its block group
mid = seg[["seg_id", "geometry"]].copy()
mid["geometry"] = seg.geometry.centroid
j = gpd.sjoin_nearest(mid, bg[["GEOID", "geometry"]], how="left", max_distance=50).drop_duplicates("seg_id")
seg = seg.merge(j[["seg_id", "GEOID"]], on="seg_id", how="left")
seg = seg.merge(acs[["GEOID", "bg_households", "income_imputed", "income_topcoded"] + acs_cols],
                on="GEOID", how="left")
print("Segments linked to a block group:", seg["GEOID"].notna().mean().round(3))

# Environmental-justice flag, looked up by each segment's own block group ID
ej = gpd.read_file(path("ej_2020_boston_area.geojson"))
ej_ids = set(ej.loc[ej["ej"].astype(str).str.upper().isin(["Y", "YES", "1", "TRUE"]), "geoid"]
               .astype(str).str.zfill(12))
seg["is_ej"] = seg["GEOID"].astype(str).isin(ej_ids)

boston_bgs = set(seg["GEOID"].dropna())
print(f"Boston block groups that are EJ: {len(boston_bgs & ej_ids) / len(boston_bgs):.1%}")
print(f"Segments in EJ block groups: {seg['is_ej'].mean():.1%}")

Block groups with ACS data: 681 of 681
Segments linked to a block group: 1.0
Boston block groups that are EJ: 79.3%
Segments in EJ block groups: 78.9%


Cell 11: weather features and save the clean data

In [56]:
weather = pd.read_csv(path("boston_daily_weather.csv"), parse_dates=["time"])
weather["freeze_thaw"] = (weather["temperature_2m_min"] < 0) & (weather["temperature_2m_max"] > 0)
weather["hdd_65F"] = (18.33 - (weather["temperature_2m_max"] + weather["temperature_2m_min"]) / 2).clip(lower=0)
annual_wx = (weather.groupby(weather["time"].dt.year)
                    .agg(freeze_thaw_days=("freeze_thaw", "sum"), hdd=("hdd_65F", "sum"),
                         precip_mm=("precipitation_sum", "sum"))
                    .rename_axis("year").reset_index())
annual_wx.to_csv(os.path.join(PROC_DIR, "annual_weather.csv"), index=False)

seg.to_file(os.path.join(PROC_DIR, "segments_clean.geojson"), driver="GeoJSON")
print("Saved", len(seg), "segments with", seg.shape[1], "columns")
print("\nMissing values (share):")
print(seg.isna().mean().round(3)[lambda s: s > 0])
seg.drop(columns="geometry").head()

Saved 17820 segments with 27 columns

Missing values (share):
ST_TYPE     0.021
addr_min    0.098
addr_max    0.098
dtype: float64


,seg_id,st_key,ST_NAME,ST_TYPE,CFCC,NBHD_L,addr_min,addr_max,length_m,n_addresses,...,GEOID,bg_households,income_imputed,income_topcoded,median_hh_income,pct_gas_heat,pct_elec_heat,pct_pre1940,pct_renter,is_ej
0,1,A ST,A,ST,A31,SOUTH BOSTON,12.0,29.0,50.802055,4.0,...,250250612031,1386,False,True,250001.0,0.601732,0.303752,0.183226,0.379509,True
1,2,A ST,A,ST,A31,SOUTH BOSTON,24.0,43.0,49.421264,4.0,...,250250612031,1386,False,True,250001.0,0.601732,0.303752,0.183226,0.379509,True
2,3,A ST,A,ST,A31,SOUTH BOSTON,44.0,67.0,54.413617,1.0,...,250250612031,1386,False,True,250001.0,0.601732,0.303752,0.183226,0.379509,True
3,4,A ST,A,ST,A31,SOUTH BOSTON,66.0,72.0,59.997294,1.0,...,250250612031,1386,False,True,250001.0,0.601732,0.303752,0.183226,0.379509,True
4,5,A ST,A,ST,A31,SOUTH BOSTON,73.0,90.0,54.643979,3.0,...,250250612031,1386,False,True,250001.0,0.601732,0.303752,0.183226,0.379509,True
